# Multi-Objective Bayesian Optimization with BoTorch

## Breast Cancer Detection Hyperparameter Optimization

This notebook demonstrates how to use **BoTorch** with **qNEHVI acquisition** for efficient multi-objective hyperparameter optimization.

### Key Features
- **Sample Efficiency:** 200 evaluations vs 1000 for NSGA-III (80% reduction)
- **State-of-the-art:** qNEHVI (Quasi-Monte Carlo Noisy Expected Hypervolume Improvement)
- **Intelligent Exploration:** GP-based uncertainty quantification
- **Resumable:** Checkpoint every N iterations

### Problem Specification

**5 Hyperparameters:**
1. Learning rate: [1e-5, 1e-3] (log-scale)
2. Weight decay: [1e-6, 1e-2] (log-scale)
3. Dropout: [0.0, 0.5]
4. Augmentation strength: [0.0, 1.0]
5. Unfreeze fraction: [0.0, 1.0]

**4 Objectives (all minimization):**
1. -PR-AUC (maximize PR-AUC)
2. -AUROC (maximize AUROC)
3. Brier score (minimize)
4. Cross-dataset degradation (minimize)

## 1. Setup and Imports

In [37]:
# !pip install -q torch==2.8.0 torchvision==0.23.0 \
#     --index-url https://download.pytorch.org/whl/cu126

# !pip install -q -r requirements.txt

In [38]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import yaml


ENV = "local_cpu"
# ENV = "kaggle_gpu"

CONFIG_DIR = Path("breast_cancer_detection/configs")

def deep_update(base, override):
    for key, value in override.items():
        if (
            key in base
            and isinstance(base[key], dict)
            and isinstance(value, dict)
        ):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_config(base_path, env_path):
    with open(base_path, "r") as f:
        config = yaml.safe_load(f)

    with open(env_path, "r") as f:
        override = yaml.safe_load(f)

    return deep_update(config, override)


config = load_config(
    CONFIG_DIR / "base.yaml",
    CONFIG_DIR / f"{ENV}.yaml"
)


project_root = Path(
    config["paths"]["project_root"]
).resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


print("Environment:", ENV)
print("Project root:", project_root)
print("Config directory:", CONFIG_DIR)

Environment: local_cpu
Project root: C:\Users\HP\Downloads\UI MSC WORK
Config directory: breast_cancer_detection\configs


In [39]:
# Import BoTorch modules
from breast_cancer_detection.src.evaluation_functions import (
    create_evaluation_function, 
    decode_hyperparameters
)
from breast_cancer_detection.src.botorch_mobo import (
    MultiObjectiveGPModel,
    qNEHVIAcquisition,
    compute_reference_point,
    initial_sobol_sampling
)
from breast_cancer_detection.src.botorch_utils import (
    HyperparameterTransform,
    BoTorchCheckpoint
)
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor
from breast_cancer_detection.src.datasets import (
    VinDRMammoBinaryDataset,
    create_breast_level_splits
)

# Import multi-GPU evaluation module
from breast_cancer_detection.src.multigpu_evaluation import (
    evaluate_batch_parallel,
    evaluate_batch_sequential,
    detect_gpus,
    set_multiprocessing_start_method
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports successful
PyTorch version: 2.8.0+cpu
CUDA available: False


In [40]:
# ========================================
# COMPREHENSIVE HOTFIX - Patches all bugs in Kaggle dataset
# ========================================
import torch

print("Applying runtime patches for bugs in Kaggle input dataset...")

# ---------------------------------------------------
# PATCH 1: Fix evaluate_batch_sequential
# ---------------------------------------------------
def evaluate_batch_sequential_fixed(evaluate_fn, X_batch, transform, verbose=True, save_callback=None):
    """Fixed version - dropout_rate -> dropout + cross-dataset degradation + incremental saving"""
    batch_size = X_batch.shape[0]

    if verbose:
        print(f"\nSequential evaluation: {batch_size} candidates")

    Y_batch = []
    hyperparams_list = []

    for i in range(batch_size):
        x = X_batch[i].cpu().numpy()
        hyperparams = transform.to_real_hyperparams(x)

        if verbose:
            print(f"\nEvaluating candidate {i+1}/{batch_size}")
            print(f"  LR: {hyperparams['learning_rate']:.6f}")
            print(f"  Weight decay: {hyperparams['weight_decay']:.6f}")
            print(f"  Dropout: {hyperparams['dropout']:.3f}")  # FIXED

        try:
            metrics = evaluate_fn(hyperparams)

            objectives = [
                -metrics["pr_auc"],
                -metrics["auroc"],
                metrics["brier"],
                metrics["cross_dataset_degradation"]  # UPDATED
            ]

            Y_batch.append(objectives)
            hyperparams_list.append(hyperparams)

            if verbose:
                print(f"  PR-AUC: {metrics['pr_auc']:.4f}")
                print(f"  AUROC: {metrics['auroc']:.4f}")
                print(f"  Brier: {metrics['brier']:.4f}")
                print(f"  Cross-dataset deg: {metrics['cross_dataset_degradation']:.4f}")  # UPDATED
            
            # INCREMENTAL SAVE: Save immediately after each evaluation
            if save_callback is not None:
                save_callback(hyperparams, metrics)

        except Exception as exc:
            print(f"Candidate {i+1} generated an exception: {exc}")
            import traceback
            traceback.print_exc()
            objectives = [float('nan')] * 4
            Y_batch.append(objectives)
            hyperparams_list.append(hyperparams)

    Y_batch = torch.tensor(Y_batch, dtype=torch.float64)
    return Y_batch, hyperparams_list

# ---------------------------------------------------
# PATCH 2: Fix train_and_evaluate (verbose=True + inbreast_calibration_dataset)
# ---------------------------------------------------
def train_and_evaluate_fixed(
    hyperparams,
    train_dataset,
    val_dataset,
    device,
    batch_size=4,
    num_workers=2,
    patience=10,
    max_epochs=100,
    pos_weight=None,
    random_seed=42,
    inbreast_calibration_dataset=None  # NEW
):
    """Fixed version - verbose=True + inbreast_calibration_dataset parameter"""
    import numpy as np
    from torch.utils.data import DataLoader, Subset
    from breast_cancer_detection.src.models import build_resnet152
    from breast_cancer_detection.src.augmentations import get_augmentation
    from breast_cancer_detection.src.training import train_model
    from breast_cancer_detection.src.datasets import VinDRMammoBinaryDataset
    
    # Set random seed
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # Create augmentation transform
    augmentation = get_augmentation(hyperparams["augmentation_strength"])

    # Handle augmentation for Subset datasets
    if isinstance(train_dataset, Subset):
        base_dataset = train_dataset.dataset
        train_indices = train_dataset.indices

        if isinstance(base_dataset, VinDRMammoBinaryDataset):
            train_dataset_aug = VinDRMammoBinaryDataset(
                images_root=base_dataset.images_root,
                csv_file=None,
                preprocessor=base_dataset.preprocessor,
                transform=augmentation,
                samples=base_dataset.samples
            )
            train_dataset_aug = Subset(train_dataset_aug, train_indices)
        else:
            base_dataset.transform = augmentation
            train_dataset_aug = train_dataset
    else:
        train_dataset.transform = augmentation
        train_dataset_aug = train_dataset

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset_aug,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    # Build model
    model = build_resnet152(
        pretrained=True,
        dropout=hyperparams["dropout"],
        unfreeze_fraction=hyperparams["unfreeze_fraction"]
    )
    model = model.to(device)

    # Train model
    model, metrics = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        val_dataset=val_dataset,
        device=device,
        learning_rate=hyperparams["learning_rate"],
        weight_decay=hyperparams["weight_decay"],
        patience=patience,
        max_epochs=max_epochs,
        pos_weight=pos_weight,
        verbose=True,  # FIXED: was False
        inbreast_calibration_dataset=inbreast_calibration_dataset  # NEW
    )

    return metrics

# ---------------------------------------------------
# Apply patches
# ---------------------------------------------------
from breast_cancer_detection.src import multigpu_evaluation
from breast_cancer_detection.src import evaluation_functions

# Patch the functions
multigpu_evaluation.evaluate_batch_sequential = evaluate_batch_sequential_fixed
evaluation_functions.train_and_evaluate = train_and_evaluate_fixed

print("✓ Patch 1: evaluate_batch_sequential - fixed 'dropout_rate' -> 'dropout'")
print("✓ Patch 2: train_and_evaluate - fixed 'verbose=False' -> 'verbose=True'")
print("✓ Patch 3: Added inbreast_calibration_dataset parameter")
print("✓ Patch 4: Updated metric names: robustness_degradation -> cross_dataset_degradation")
print("✓ All patches applied successfully!")
print("\nYou can now proceed with optimization.")

Applying runtime patches for bugs in Kaggle input dataset...
✓ Patch 1: evaluate_batch_sequential - fixed 'dropout_rate' -> 'dropout'
✓ Patch 2: train_and_evaluate - fixed 'verbose=False' -> 'verbose=True'
✓ Patch 3: Added inbreast_calibration_dataset parameter
✓ Patch 4: Updated metric names: robustness_degradation -> cross_dataset_degradation
✓ All patches applied successfully!

You can now proceed with optimization.


## 2. Configuration

In [41]:
# ========================================
# CONFIGURATION
# ========================================

# Data paths (Kaggle)
DATA_ROOT = config["paths"]["vindr_images"]
CSV_FILE = config["paths"]["vindr_csv"]



# Optimization parameters
N_INITIAL = config["optimization"]["n_initial"]        # Initial Sobol samples (2× dimensionality)
N_ITERATIONS = config["optimization"]["n_iterations"]     # BO iterations after initial sampling
BATCH_SIZE = config["optimization"]["candidates_per_iteration"]        # Candidates per BO iteration
TOTAL_BUDGET = N_INITIAL + N_ITERATIONS * BATCH_SIZE  # = 56

# Training parameters
TRAIN_BATCH_SIZE = config["training"]["batch_size"]
MAX_EPOCHS = config["training"]["max_epochs"]
PATIENCE = config["training"]["patience"]
NUM_WORKERS = config["training"]["num_workers"]

# Acquisition parameters
ACQ_SAMPLES = 128
ACQ_RESTARTS = 20
REF_POINT_OFFSET = 0.1

# Checkpointing
CHECKPOINT_FREQ = 5

# Output directory - Use /kaggle/working/ on Kaggle for write access
OUTPUT_DIR = Path(
    config["paths"]["output_dir"]
)

RUN_ID = (
    f"{ENV}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
# Hardware
requested_device = config["environment"]["device"]

if (
    requested_device == "cuda"
    and torch.cuda.is_available()
):
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

SEED = config["experiment"]["seed"]

# Resume from checkpoint (set to None for fresh start)
RESUME_FROM = None  # e.g., "/kaggle/working/results/botorch_mobo/run_001/checkpoint_iter10.pt"

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)

print(f"Environment:     {ENV}")
print(f"Device:          {DEVICE}")

print("\nOptimization:")
print(f"  Initial:       {N_INITIAL}")
print(f"  BO iterations: {N_ITERATIONS}")
print(f"  BO batch:      {BATCH_SIZE}")
print(f"  Total budget:  {TOTAL_BUDGET}")

print("\nTraining:")
print(f"  Batch size:    {TRAIN_BATCH_SIZE}")
print(f"  Max epochs:    {MAX_EPOCHS}")
print(f"  Patience:      {PATIENCE}")
print(f"  Workers:       {NUM_WORKERS}")

print("\nPaths:")
print(f"  VinDr images:  {DATA_ROOT}")
print(f"  VinDr CSV:     {CSV_FILE}")
print(f"  Output:        {OUTPUT_DIR / RUN_ID}")

print("=" * 80)

CONFIGURATION
Environment:     local_cpu
Device:          cpu

Optimization:
  Initial:       1
  BO iterations: 1
  BO batch:      1
  Total budget:  2

Training:
  Batch size:    1
  Max epochs:    2
  Patience:      1
  Workers:       0

Paths:
  VinDr images:  data/vindr/vindr_mammo_dataset_dave/images
  VinDr CSV:     data/vindr/vindr_mammo_dataset_dave/metadata/stratified_selection.csv
  Output:        results\botorch_mobo\local_cpu_20260813_155237


In [42]:
def resume_from_csv(csv_path, transform, output_dir_path):
    """
    Reconstruct optimization state from evaluations.csv

    Returns:
        X_train: Tensor of evaluated hyperparameters in normalized space
        Y_train: Tensor of objectives
        all_hyperparams: List of hyperparameter dicts
        start_iter: Starting iteration for BO loop
    """
    import pandas as pd
    import torch
    import numpy as np
    from datetime import datetime

    print("="*80)
    print("RESUMING FROM CSV")
    print("="*80)

    # 1. Load CSV
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    n_evals = len(df)

    print(f"\n✓ Loaded CSV: {csv_path}")
    print(f"  Total evaluations: {n_evals}")

    # 2. Verify CSV has required columns
    # Handle backward compatibility: old column name was 'robustness', new is 'cross_dataset_degradation'
    required_cols = [
        'learning_rate', 'weight_decay', 'dropout',
        'augmentation_strength', 'unfreeze_fraction',
        'pr_auc', 'auroc', 'brier'
    ]
    
    # Check for degradation column (support both old and new names)
    degradation_col = None
    if 'cross_dataset_degradation' in df.columns:
        degradation_col = 'cross_dataset_degradation'
        print("✓ Using 'cross_dataset_degradation' column")
    elif 'robustness' in df.columns:
        degradation_col = 'robustness'
        print("✓ Using 'robustness' column (old name)")
    elif 'robustness_degradation' in df.columns:
        degradation_col = 'robustness_degradation'
        print("✓ Using 'robustness_degradation' column (old name)")
    else:
        raise ValueError(
            f"CSV missing degradation column! Expected one of: "
            f"'cross_dataset_degradation', 'robustness', 'robustness_degradation'"
        )
    
    # Check other required columns
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"CSV missing required columns: {missing}")

    print(f"✓ All required columns present")

    # 3. Reconstruct hyperparameters
    all_hyperparams = []
    X_linear = []

    for _, row in df.iterrows():
        # Real hyperparameters
        hyperparams = {
            'learning_rate': row['learning_rate'],
            'weight_decay': row['weight_decay'],
            'dropout': row['dropout'],
            'augmentation_strength': row['augmentation_strength'],
            'unfreeze_fraction': row['unfreeze_fraction']
        }
        all_hyperparams.append(hyperparams)

        # Convert to linear space manually (log-scale for LR and WD)
        x_linear = [
            np.log10(hyperparams['learning_rate']),      # LR: log-scale
            np.log10(hyperparams['weight_decay']),       # WD: log-scale
            hyperparams['dropout'],                       # Dropout: linear
            hyperparams['augmentation_strength'],         # Aug: linear
            hyperparams['unfreeze_fraction']              # Unfreeze: linear
        ]
        X_linear.append(x_linear)

    # Convert to tensor in linear space
    X_linear_tensor = torch.tensor(X_linear, dtype=torch.float64)
    
    # Normalize to [0,1] using transform
    X_train = transform.to_normalized(X_linear_tensor)

    print(f"✓ Reconstructed X_train: {X_train.shape}")

    # 4. Reconstruct objectives (use detected column name)
    Y_train = torch.tensor([
        [
            -row['pr_auc'],                          # Objective 1: -PR-AUC (minimize)
            -row['auroc'],                           # Objective 2: -AUROC (minimize)
            row['brier'],                            # Objective 3: Brier (minimize)
            row[degradation_col]                     # Objective 4: Degradation (minimize)
        ]
        for _, row in df.iterrows()
    ], dtype=torch.float64)

    print(f"✓ Reconstructed Y_train: {Y_train.shape}")

    # 5. Determine start_iter (which BO iteration to resume from)
    if n_evals < N_INITIAL:
        raise ValueError(
            f"CSV has only {n_evals} evaluations, but N_INITIAL={N_INITIAL}. "
            f"Initial sampling incomplete - cannot resume."
        )

    # Calculate completed BO iterations
    bo_evals = n_evals - N_INITIAL  # Evaluations from BO phase
    completed_bo_iters = bo_evals // BATCH_SIZE  # Complete BO iterations

    # Resume from next iteration
    start_iter = completed_bo_iters

    print(f"\n{'='*80}")
    print(f"PROGRESS ANALYSIS")
    print(f"{'='*80}")
    print(f"  Initial sampling:     {N_INITIAL} evals (complete)")
    print(f"  BO evaluations:       {bo_evals} evals")
    print(f"  Completed BO iters:   {completed_bo_iters}/{N_ITERATIONS}")
    print(f"  Resume from iter:     {start_iter}")

    # Check if there are partial evaluations
    partial_evals = bo_evals % BATCH_SIZE
    if partial_evals > 0:
        print(f"\n⚠ WARNING: {partial_evals} partial evaluations detected!")
        print(f"  Iteration {start_iter} had {partial_evals}/{BATCH_SIZE} candidates")
        print(f"  These will be DISCARDED for safety")
        print(f"  Iteration {start_iter} will re-run all {BATCH_SIZE} candidates")

        # Remove partial evaluations
        complete_evals = N_INITIAL + (completed_bo_iters * BATCH_SIZE)
        X_train = X_train[:complete_evals]
        Y_train = Y_train[:complete_evals]
        all_hyperparams = all_hyperparams[:complete_evals]

        print(f"  Trimmed to {complete_evals} complete evaluations")
    else:
        print(f"✓ No partial evaluations - clean state")

    print(f"{'='*80}\n")

    # 6. Save backup CSV
    backup_path = output_dir_path / f'evaluations_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(backup_path, index=False)
    print(f"✓ Backup saved: {backup_path}")

    # 7. Summary
    print(f"\n{'='*80}")
    print(f"RESUME SUMMARY")
    print(f"{'='*80}")
    print(f"  Total evaluations:    {len(X_train)}")
    print(f"  Starting iteration:   {start_iter}/{N_ITERATIONS}")
    print(f"  Remaining iterations: {N_ITERATIONS - start_iter}")
    print(f"  Remaining evals:      {(N_ITERATIONS - start_iter) * BATCH_SIZE}")
    print(f"{'='*80}\n")

    return X_train, Y_train, all_hyperparams, start_iter

print("✓ CSV resume function defined")

✓ CSV resume function defined


In [ ]:
# ========================================
# CSV RESUME CONFIGURATION
# ========================================

# RESUME FROM CSV (when no checkpoint available)
RESUME_FROM_CSV = config["optimization"]["resume_from_csv"]  # Set to True to resume from CSV


# CSV and split info paths - UPDATE THESE when resuming!
# For Kaggle resume: Point to your input dataset
# For local resume: Point to your previous run directory
CSV_RESUME_PATH = config["paths"]["csv_resume_path"] 
INBREAST_SPLIT_INFO_PATH = config["paths"]["inbreast_split_info_path"] 

# Safety checks
if RESUME_FROM_CSV and RESUME_FROM is not None:
    raise ValueError("Cannot use both RESUME_FROM and RESUME_FROM_CSV! Choose one.")

if RESUME_FROM_CSV and CSV_RESUME_PATH is None:
    raise ValueError("RESUME_FROM_CSV is True but CSV_RESUME_PATH is not set! Please specify the path to your evaluations.csv")

print("="*80)
print("CSV RESUME CONFIGURATION")
print("="*80)
if RESUME_FROM_CSV:
    print(f"✓ CSV resume enabled")
    print(f"  CSV path: {CSV_RESUME_PATH}")
    print(f"  Split info path: {INBREAST_SPLIT_INFO_PATH}")
    if CSV_RESUME_PATH and not Path(CSV_RESUME_PATH).exists():
        print(f"  ⚠️  WARNING: CSV file not found at specified path!")
    if INBREAST_SPLIT_INFO_PATH and not Path(INBREAST_SPLIT_INFO_PATH).exists():
        print(f"  ⚠️  WARNING: Split info file not found at specified path!")
else:
    print(f"  CSV resume disabled")
print("="*80)

CSV RESUME CONFIGURATION
✓ CSV resume enabled
  CSV path: /kaggle/input/dataaa/evaluations (10).csv
  Split info path: /kaggle/input/dataaa/inbreast_split_info.json
  ⚠️  WARNING: CSV file not found at specified path!
  ⚠️  WARNING: Split info file not found at specified path!


## 2.5. Multi-Session Configuration (Kaggle Support)

**For Kaggle Users:** Configuration for multi-session training with automatic checkpoint management.

Set `USE_KAGGLE_PERSISTENCE = True` to enable automatic checkpoint upload/download to Kaggle Datasets.

In [44]:
# ========================================
# MULTI-SESSION CONFIGURATION (KAGGLE)
# ========================================

# Session management
SESSION_MAX_HOURS = 22        # Conservative: 22 hours per session
SESSION_BUFFER_MINUTES = 30   # Safety buffer before timeout
AUTO_STOP_ON_TIMEOUT = True   # Auto-save and stop before timeout

# Kaggle Dataset for checkpoints
KAGGLE_DATASET_SLUG = "yourusername/breast-cancer-mobo-checkpoints"  # UPDATE THIS
USE_KAGGLE_PERSISTENCE = False  # Set to True for Kaggle runs

# Import session management
if USE_KAGGLE_PERSISTENCE:
    try:
        from breast_cancer_detection.src.kaggle_checkpoint_manager import KaggleCheckpointManager
        from breast_cancer_detection.src.session_manager import SessionManager
        
        kaggle_manager = KaggleCheckpointManager(
            dataset_slug=KAGGLE_DATASET_SLUG,
            output_dir=OUTPUT_DIR / RUN_ID
        )
        
        session_manager = SessionManager(
            max_session_hours=SESSION_MAX_HOURS,
            buffer_minutes=SESSION_BUFFER_MINUTES
        )
        
        print("\n✓ Multi-session support enabled")
        print(f"  Dataset: {KAGGLE_DATASET_SLUG}")
        print(f"  Max session: {SESSION_MAX_HOURS} hours")
        print(f"  Safety buffer: {SESSION_BUFFER_MINUTES} minutes")
    except ImportError as e:
        print(f"\n⚠ Warning: Could not import session management modules")
        print(f"  Error: {e}")
        print(f"  Falling back to single-session mode")
        USE_KAGGLE_PERSISTENCE = False
else:
    print("\n Multi-session support disabled (USE_KAGGLE_PERSISTENCE = False)")
    print("  Running in standard single-session mode")


 Multi-session support disabled (USE_KAGGLE_PERSISTENCE = False)
  Running in standard single-session mode


In [45]:
# ========================================
# MULTI-GPU CONFIGURATION (KAGGLE)
# ========================================

# Multi-GPU settings
USE_MULTI_GPU = True  # Set to True to use 2 GPUs on Kaggle
NUM_GPUS = 2          # Number of GPUs to use

# Set multiprocessing start method (required for CUDA)
set_multiprocessing_start_method()

# Detect available GPUs
if USE_MULTI_GPU and torch.cuda.is_available():
    num_available_gpus = torch.cuda.device_count()
    
    if num_available_gpus >= 2:
        print("\n" + "="*80)
        print("MULTI-GPU MODE ENABLED")
        print("="*80)
        
        # Show GPU information
        num_gpus, gpu_names = detect_gpus(verbose=True)
        
        PARALLEL_EVALUATION = True
        
        print(f"\n Multi-GPU parallel evaluation enabled")
        print(f"  Using {min(num_available_gpus, NUM_GPUS)} GPUs")
        print(f"  Expected speedup: ~{min(num_available_gpus, NUM_GPUS)}x")
        print(f"  BATCH_SIZE optimized for {NUM_GPUS} GPUs: {BATCH_SIZE}")
        print("="*80)
    else:
        print(f"\n Only {num_available_gpus} GPU available")
        print(f"  Falling back to single GPU mode")
        PARALLEL_EVALUATION = False
else:
    if not USE_MULTI_GPU:
        print("\n Multi-GPU mode disabled (USE_MULTI_GPU = False)")
    else:
        print("\n CUDA not available - cannot use multi-GPU")
    print("  Using sequential evaluation on single device")
    PARALLEL_EVALUATION = False


Multiprocessing start method set to 'spawn' (required for CUDA)

 CUDA not available - cannot use multi-GPU
  Using sequential evaluation on single device


In [46]:
# ========================================
# VERIFY MULTI-GPU SETUP (DIAGNOSTIC)
# ========================================

print("\n" + "="*80)
print("MULTI-GPU SETUP VERIFICATION")
print("="*80)

# Check 1: CUDA availability
print(f"\n1. CUDA Check:")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   PyTorch version: {torch.__version__}")

# Check 2: GPU count
print(f"\n2. GPU Detection:")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"   Number of GPUs: {num_gpus}")
    for i in range(num_gpus):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print(f"   No GPUs detected")

# Check 3: Multi-GPU configuration
print(f"\n3. Multi-GPU Configuration:")
print(f"   USE_MULTI_GPU: {USE_MULTI_GPU}")
print(f"   NUM_GPUS: {NUM_GPUS}")
print(f"   PARALLEL_EVALUATION: {PARALLEL_EVALUATION}")

# Check 4: Expected behavior
print(f"\n4. Expected Behavior:")
if PARALLEL_EVALUATION:
    print(f"   ✓ Will use PARALLEL evaluation on {NUM_GPUS} GPUs")
    print(f"   ✓ Evaluations will run simultaneously on multiple GPUs")
    print(f"   ✓ Expected speedup: ~{NUM_GPUS}x")
else:
    print(f"   ✗ Will use SEQUENTIAL evaluation")
    print(f"   ✗ Evaluations will run one at a time")
    if not torch.cuda.is_available():
        print(f"   Reason: CUDA not available")
    elif torch.cuda.device_count() < 2:
        print(f"   Reason: Only {torch.cuda.device_count()} GPU(s) available")
    elif not USE_MULTI_GPU:
        print(f"   Reason: USE_MULTI_GPU is False")

print("\n" + "="*80)

# Quick test if GPUs available
if PARALLEL_EVALUATION:
    print("\nQuick GPU Test:")
    print("  Testing GPU access from main process...")
    try:
        for i in range(NUM_GPUS):
            device = torch.device(f'cuda:{i}')
            test_tensor = torch.ones(10, device=device)
            print(f"  ✓ GPU {i} accessible: {test_tensor.device}")
    except Exception as e:
        print(f"  ✗ GPU test failed: {e}")
    print("="*80)



MULTI-GPU SETUP VERIFICATION

1. CUDA Check:
   CUDA available: False

2. GPU Detection:
   No GPUs detected

3. Multi-GPU Configuration:
   USE_MULTI_GPU: True
   NUM_GPUS: 2
   PARALLEL_EVALUATION: False

4. Expected Behavior:
   ✗ Will use SEQUENTIAL evaluation
   ✗ Evaluations will run one at a time
   Reason: CUDA not available



## 3. Load Dataset

Load VinDr-Mammo with breast-level split.

In [47]:
def setup_dataset(data_root, csv_file, seed=42):
    """Setup VinDr-Mammo dataset with breast-level split."""
    print("Loading VinDr-Mammo dataset...")
    
    preprocessor = MammographyPreprocessor(
        target_size=(720, 480),
        aspect_ratio=1.5
    )
    
    dataset = VinDRMammoBinaryDataset(
        images_root=data_root,
        csv_file=csv_file,
        preprocessor=preprocessor
    )
    
    print(f"Total samples: {len(dataset)}")
    
    # Class counts
    all_labels = [dataset[i][1].item() for i in range(len(dataset))]
    n_benign = sum([1 for l in all_labels if l == 0])
    n_malignant = sum([1 for l in all_labels if l == 1])
    
    print(f"  Benign: {n_benign}")
    print(f"  Malignant: {n_malignant}")
    print(f"  Ratio: {n_benign/n_malignant:.2f}:1")
    
    # Breast-level split
    print("\nCreating breast-level train/val split...")
    train_dataset, val_dataset = create_breast_level_splits(
        dataset=dataset,
        train_ratio=0.8,
        random_state=seed,
        stratify=True
    )
    
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Val: {len(val_dataset)} samples")
    
    return train_dataset, val_dataset, n_benign, n_malignant

# Load data
train_dataset, val_dataset, n_benign, n_malignant = setup_dataset(
    DATA_ROOT, CSV_FILE, SEED
)

pos_weight = n_benign / n_malignant
print(f"\nPos weight: {pos_weight:.3f}")

Loading VinDr-Mammo dataset...
[VinDr-Mammo] Loaded 636 image-level samples
Total samples: 636
  Benign: 410
  Malignant: 226
  Ratio: 1.81:1

Creating breast-level train/val split...

[Breast-level split]
  Total breasts: 318
  Train breasts: 254 (508 images)
  Val breasts: 64 (128 images)
  Train label distribution: [164  90]
  Val label distribution: [41 23]
  Train: 508 samples
  Val: 128 samples

Pos weight: 1.814


In [48]:
# ========================================
# LOAD INBREAST DATASET FROM .XLS FILE
# ========================================

from breast_cancer_detection.src import (
    INbreastDatasetFromXLS,
    create_inbreast_calibration_test_splits
)

print("\n" + "="*80)
print("LOADING INBREAST FROM .XLS FILE")
print("="*80)

# Paths - UPDATE THESE for your environment
INBREAST_DICOM_DIR = Path(
    config["paths"]["inbreast_dicom_dir"]
)

INBREAST_XLS_FILE = Path(
    config["paths"]["inbreast_metadata"]
)


print(f"\nPaths:")
print(f"  DICOM directory: {INBREAST_DICOM_DIR}")
print(f"  XLS file: {INBREAST_XLS_FILE}")

# Check if files exist
if not os.path.exists(INBREAST_DICOM_DIR):
    print(f"  ✗ DICOM directory not found: {INBREAST_DICOM_DIR}")
    print(f"  ⚠  Skipping INbreast loading - cross-dataset degradation will be 0.0")
    inbreast_calibration = None
    inbreast_test = None
elif not os.path.exists(INBREAST_XLS_FILE):
    print(f"  ✗ XLS file not found: {INBREAST_XLS_FILE}")
    print(f"  ⚠  Skipping INbreast loading - cross-dataset degradation will be 0.0")
    inbreast_calibration = None
    inbreast_test = None
else:
    print("  ✓ Files found")

    # Get preprocessor from VinDr dataset
    if isinstance(train_dataset, torch.utils.data.Subset):
        preprocessor = train_dataset.dataset.preprocessor
    else:
        preprocessor = train_dataset.preprocessor

    # Load INbreast dataset from .xls
    inbreast_dataset = INbreastDatasetFromXLS(
        dicom_dir=INBREAST_DICOM_DIR,
        xls_file=INBREAST_XLS_FILE,
        preprocessor=preprocessor  # Reuse from VinDr
    )

    print(f"\n✓ INbreast dataset loaded")
    print(f"  Total images: {len(inbreast_dataset)}")

    # Get breast groups
    breast_groups = inbreast_dataset.get_breast_groups()
    print(f"  Total breasts: {len(breast_groups)}")

    # Show breast grouping statistics
    images_per_breast = [len(g['image_indices']) for g in breast_groups]
    from collections import Counter
    distribution = Counter(images_per_breast)

    print(f"\n  Images per breast distribution:")
    for n_images, count in sorted(distribution.items()):
        print(f"    {n_images} images: {count} breasts")

    # ========================================
    # SMART SPLIT HANDLING (RESUME-AWARE)
    # ========================================
    print("\n" + "-"*80)
    print("Creating/Loading 20/80 calibration/test split...")
    print("-"*80)

    import json
    
    # Create output directory early (needed for split info)
    temp_output_dir = OUTPUT_DIR / RUN_ID
    temp_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Determine split info path
    # Priority: 1) User-specified path (for resume from input)
    #           2) Output directory path (for fresh start or local resume)
    if RESUME_FROM_CSV and INBREAST_SPLIT_INFO_PATH is not None:
        split_info_path = Path(INBREAST_SPLIT_INFO_PATH)
        print(f"\n✓ Using user-specified split info path: {split_info_path}")
    else:
        split_info_path = temp_output_dir / 'inbreast_split_info.json'
        print(f"\n✓ Using default split info path: {split_info_path}")

    if split_info_path.exists():
        # RESUME MODE: Load existing split to ensure consistency
        print("\n" + "="*80)
        print("⚠️  EXISTING SPLIT DETECTED (RESUME MODE)")
        print("="*80)
        
        with open(split_info_path, 'r') as f:
            split_info = json.load(f)
        
        print(f"\n✓ Loaded existing split configuration")
        print(f"  Split source: {split_info_path}")
        print(f"  Random seed: {split_info['random_state']}")
        print(f"  Original creation: Using saved random state")
        
        # Recreate SAME split using saved random seed
        inbreast_calibration, inbreast_test, recreated_split_info = create_inbreast_calibration_test_splits(
            dataset=inbreast_dataset,
            calibration_ratio=0.2,
            random_state=split_info['random_state'],  # CRITICAL: Use SAME seed
            stratify=True
        )
        
        # Verify splits match
        if recreated_split_info['n_breasts_calibration'] != split_info['n_breasts_calibration']:
            print(f"\n⚠️  WARNING: Split mismatch detected!")
            print(f"  Expected {split_info['n_breasts_calibration']} calibration breasts")
            print(f"  Got {recreated_split_info['n_breasts_calibration']} calibration breasts")
            raise ValueError("Split recreation failed - data may have changed!")
        
        print(f"✓ Split verification passed - same split as previous run")
        print(f"\n⚠️  CRITICAL: Using SAME calibration set as previous evaluations")
        print(f"   This ensures cross-dataset degradation values are comparable!")
        
        # Save a copy to new output directory (for future resumes from this run)
        new_split_path = temp_output_dir / 'inbreast_split_info.json'
        if new_split_path != split_info_path:
            with open(new_split_path, 'w') as f:
                json.dump(split_info, f, indent=2)
            print(f"\n✓ Copied split info to new output directory: {new_split_path}")
        
    else:
        # FRESH START: Create new split
        print("\n" + "="*80)
        print("CREATING NEW SPLIT (FRESH START)")
        print("="*80)
        
        inbreast_calibration, inbreast_test, split_info = create_inbreast_calibration_test_splits(
            dataset=inbreast_dataset,
            calibration_ratio=0.2,
            random_state=SEED,
            stratify=True
        )

        # Save split info for reproducibility
        new_split_path = temp_output_dir / 'inbreast_split_info.json'
        with open(new_split_path, 'w') as f:
            json.dump(split_info, f, indent=2)

        print(f"\n{'='*80}")
        print("⚠️  CRITICAL FILE SAVED FOR FUTURE RESUMES & ZERO-SHOT EVALUATION")
        print(f"{'='*80}")
        print(f"Split info saved to: {new_split_path}")
        print(f"\n⚠️  IMPORTANT:")
        print(f"   • This file will be reused if you resume from CSV")
        print(f"   • This file must be provided to zero-shot notebook")
        print(f"   • Upload this file as a Kaggle dataset for future resumes")
        print(f"{'='*80}\n")

    # Display split summary
    print("\n" + "="*80)
    print("INBREAST SPLIT SUMMARY")
    print("="*80)
    print(f"Total breasts:        {split_info['n_breasts_total']}")
    print(f"Calibration breasts:  {split_info['n_breasts_calibration']} ({100*split_info['n_breasts_calibration']/split_info['n_breasts_total']:.1f}%)")
    print(f"Test breasts:         {split_info['n_breasts_test']} ({100*split_info['n_breasts_test']/split_info['n_breasts_total']:.1f}%)")
    print(f"Calibration images:   {split_info['n_images_calibration']}")
    print(f"Test images:          {split_info['n_images_test']}")
    print(f"Random seed:          {split_info['random_state']}")
    print("="*80)

    print("\n⚠ IMPORTANT:")
    print("  • Calibration set will be used for Objective 4 (cross-dataset degradation)")
    print("  • Test set is HELD OUT until final evaluation")
    print("  • DO NOT use test set during hyperparameter optimization")
    print("  • When resuming, the SAME split will be automatically loaded")

    print("\n" + "="*80)
    print("INBREAST READY FOR HPO")
    print("="*80)


LOADING INBREAST FROM .XLS FILE

Paths:
  DICOM directory: data\inbreast\INbreast_Release\AllDICOMs
  XLS file: data\inbreast\INbreast.xls
  ✗ XLS file not found: data\inbreast\INbreast.xls
  ⚠  Skipping INbreast loading - cross-dataset degradation will be 0.0


# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

# Create evaluation function
print("Creating evaluation function...")
evaluate_fn = create_evaluation_function(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=DEVICE,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=2,
    patience=PATIENCE,
    max_epochs=MAX_EPOCHS,
    pos_weight=pos_weight,
    random_seed=SEED,
    inbreast_calibration_dataset=inbreast_calibration  # NEW: Cross-dataset degradation
)

print("✓ Evaluation function ready")
print(f"  Each evaluation trains a ResNet152 CNN (~30 min on GPU)")
print(f"\n✓ Evaluation objectives configured:")
print(f"  1. Maximize PR-AUC (VinDr validation)")
print(f"  2. Maximize AUROC (VinDr validation)")
print(f"  3. Minimize Brier score (VinDr validation)")
if inbreast_calibration is not None:
    print(f"  4. Minimize cross-dataset degradation (VinDr → INbreast)")
    print(f"     Using {len(inbreast_calibration)} INbreast calibration images")
else:
    print(f"  4. Cross-dataset degradation disabled (INbreast not loaded)")
    print(f"     Degradation will be 0.0 for all evaluations")

## 4. Initialize Evaluation Function

Create the evaluation function that trains CNN and returns objectives.

In [49]:
# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

# Create evaluation function
print("Creating evaluation function...")
evaluate_fn = create_evaluation_function(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=DEVICE,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=2,
    patience=PATIENCE,
    max_epochs=MAX_EPOCHS,
    pos_weight=pos_weight,
    random_seed=SEED,
    inbreast_calibration_dataset=inbreast_calibration  # FIXED: Added this parameter
)

print("✓ Evaluation function ready")
print(f"  Each evaluation trains a ResNet152 CNN (~30 min on GPU)")
print(f"\n✓ Evaluation objectives configured:")
print(f"  1. Maximize PR-AUC (VinDr validation)")
print(f"  2. Maximize AUROC (VinDr validation)")
print(f"  3. Minimize Brier score (VinDr validation)")
if inbreast_calibration is not None:
    print(f"  4. Minimize cross-dataset degradation (VinDr → INbreast)")
    print(f"     Using {len(inbreast_calibration)} INbreast calibration images")
else:
    print(f"  4. Cross-dataset degradation disabled (INbreast not loaded)")
    print(f"     Degradation will be 0.0 for all evaluations")

Creating evaluation function...
✓ Evaluation function ready
  Each evaluation trains a ResNet152 CNN (~30 min on GPU)

✓ Evaluation objectives configured:
  1. Maximize PR-AUC (VinDr validation)
  2. Maximize AUROC (VinDr validation)
  3. Minimize Brier score (VinDr validation)
  4. Cross-dataset degradation disabled (INbreast not loaded)
     Degradation will be 0.0 for all evaluations


## 5. Initialize BoTorch Components

### 🐛 Bug Fixes Applied

**Critical bugs fixed in this notebook:**

1. **Cell 16 - Bounds Fix** ⚠️ CRITICAL
   - **Was:** `bounds_linear = transform.normalized_bounds` (returns [0,1] bounds)
   - **Now:** `bounds_linear = transform.bounds_linear` (returns log-scale bounds [-5,-3] for LR, etc.)
   - **Impact:** Without this fix, learning rates were 1000x too large (e.g., 5.37 instead of 0.000537), causing training to diverge

2. **Cell 18 - Sequential Evaluation**
   - Multi-GPU parallel disabled due to closure pickling issues
   - Using sequential evaluation (slower but reliable)
   - Future: Refactor to pass dataset paths instead of dataset objects

3. **Source Code Fixes** (already applied in `breast_cancer_detection/src/`)
   - `multigpu_evaluation.py` line 176, 240: Fixed `dropout_rate` → `dropout` KeyError
   - `evaluation_functions.py` line 163: Changed `verbose=False` → `verbose=True` to show training progress

**Verification:**
- Run the verification test cell below to confirm all fixes are working
- Delete the verification cell after successful test

In [50]:
# Step 1: Create hyperparameter transform
# This handles the log-scale transformation for learning rate and weight decay
transform = HyperparameterTransform()

print("✓ Hyperparameter transform initialized")
print("  Transforms between normalized [0,1] space (for BoTorch) and actual hyperparameter values")
print(f"  Internal bounds (log-space):")
print(f"    Lower: {transform.bounds_linear[0].tolist()}")
print(f"    Upper: {transform.bounds_linear[1].tolist()}")

✓ Hyperparameter transform initialized
  Transforms between normalized [0,1] space (for BoTorch) and actual hyperparameter values
  Internal bounds (log-space):
    Lower: [-5.0, -6.0, 0.0, 0.0, 0.0]
    Upper: [-3.0, -2.0, 0.5, 1.0, 1.0]


In [51]:
# Step 2: Get bounds for BoTorch optimization
# BoTorch will work in the linear (log) space defined by transform.bounds_linear
bounds_linear = transform.bounds_linear

print("✓ Bounds defined for BoTorch (in log-space for lr/wd):")
print(f"  Lower: {bounds_linear[0].tolist()}")
print(f"  Upper: {bounds_linear[1].tolist()}")
print(f"  LR log-scale: [{bounds_linear[0][0]:.1f}, {bounds_linear[1][0]:.1f}] → real: [1e{bounds_linear[0][0]:.0f}, 1e{bounds_linear[1][0]:.0f}]")
print(f"  WD log-scale: [{bounds_linear[0][1]:.1f}, {bounds_linear[1][1]:.1f}] → real: [1e{bounds_linear[0][1]:.0f}, 1e{bounds_linear[1][1]:.0f}]")

✓ Bounds defined for BoTorch (in log-space for lr/wd):
  Lower: [-5.0, -6.0, 0.0, 0.0, 0.0]
  Upper: [-3.0, -2.0, 0.5, 1.0, 1.0]
  LR log-scale: [-5.0, -3.0] → real: [1e-5, 1e-3]
  WD log-scale: [-6.0, -2.0] → real: [1e-6, 1e-2]


In [52]:
# Step 3: Initialize GP model for Bayesian Optimization
# This model will be fitted to the evaluation data as we collect it
gp_model = MultiObjectiveGPModel(
    n_objectives=4,        # 4 objectives
    n_vars=5,              # 5 hyperparameters
    bounds=bounds_linear
)

print("✓ Multi-objective GP model initialized")
print(f"  Input dimension: 5 (hyperparameters)")
print(f"  Output dimension: 4 (objectives)")
print(f"  Model: Independent GPs for each objective")

✓ Multi-objective GP model initialized
  Input dimension: 5 (hyperparameters)
  Output dimension: 4 (objectives)
  Model: Independent GPs for each objective


In [53]:
def evaluate_batch(evaluate_fn, X_batch, transform, save_callback=None):
    """
    Evaluate a batch of hyperparameter configurations.
    
    Uses sequential evaluation (parallel has pickling issues with closures).
    
    Args:
        evaluate_fn: Evaluation function (for sequential mode)
        X_batch: (batch_size, n_vars) tensor in linear space
        transform: HyperparameterTransform instance
        save_callback: Optional function to call after each evaluation for incremental saving
        
    Returns:
        Y_batch: (batch_size, n_objs) tensor of objectives
        hyperparams_list: List of hyperparameter dicts
    """
    batch_size = X_batch.shape[0]
    
    # Note: Parallel evaluation disabled due to closure pickling issues
    # The create_evaluation_function returns a closure that captures datasets
    # which cannot be pickled for multiprocessing
    
    if PARALLEL_EVALUATION and batch_size > 1:
        print(f"\n⚠ Multi-GPU parallel evaluation disabled (closure pickling issue)")
        print(f"  Using sequential evaluation instead")
        print(f"  Batch size: {batch_size} candidates")
    elif batch_size > 1:
        print(f"\nUsing sequential evaluation")
        print(f"  Batch size: {batch_size} candidates")
    
    # Sequential evaluation with incremental saving
    Y_batch, hyperparams_list = evaluate_batch_sequential(
        evaluate_fn=evaluate_fn,
        X_batch=X_batch,
        transform=transform,
        verbose=True,
        save_callback=save_callback
    )
    
    return Y_batch, hyperparams_list

## 6. Evaluation Helper Function

## 6.5. Time Estimation

**IMPORTANT:** Review the estimated time before starting optimization!

In [54]:
# ========================================
# TIME ESTIMATION FOR OPTIMIZATION
# ========================================

import warnings
from datetime import timedelta

print("="*80)
print("TIME ESTIMATION")
print("="*80)

# Estimation parameters
# Based on empirical observations from CLAUDE.md requirements
# - Training epochs: up to MAX_EPOCHS with early stopping
# - Average early stopping: ~20-30 epochs (assuming PATIENCE=10)
# - ResNet152 on GPU: ~1-2 min per epoch (batch_size=4)
# - Robustness evaluation: runs every 5 epochs (~2 min extra)
# - Data loading, setup overhead: ~2-3 min per evaluation

if DEVICE.type == 'cuda':
    # GPU estimates
    MIN_PER_EVAL = 15  # Optimistic: fast convergence, good GPU
    AVG_PER_EVAL = 30  # Realistic: average case
    MAX_PER_EVAL = 60  # Pessimistic: slow convergence, full MAX_EPOCHS
    print("Device: GPU (CUDA)")
else:
    # CPU estimates (much slower)
    MIN_PER_EVAL = 60
    AVG_PER_EVAL = 120
    MAX_PER_EVAL = 180
    print("Device: CPU")
    warnings.warn(
        "CPU training will be VERY slow! Highly recommend using GPU.",
        UserWarning
    )

print(f"Estimated time per evaluation:")
print(f"  Optimistic:  {MIN_PER_EVAL} minutes")
print(f"  Realistic:   {AVG_PER_EVAL} minutes (expected)")
print(f"  Pessimistic: {MAX_PER_EVAL} minutes")

# Phase 1: Initial Sobol sampling
initial_min = N_INITIAL * MIN_PER_EVAL
initial_avg = N_INITIAL * AVG_PER_EVAL
initial_max = N_INITIAL * MAX_PER_EVAL

print(f"" + "="*80)
print("PHASE 1: Initial Sobol Sampling")
print("="*80)
print(f"Evaluations: {N_INITIAL}")
print(f"Estimated time:")
print(f"  Optimistic:  {initial_min:>4} minutes = {initial_min/60:.1f} hours")
print(f"  Realistic:   {initial_avg:>4} minutes = {initial_avg/60:.1f} hours")
print(f"  Pessimistic: {initial_max:>4} minutes = {initial_max/60:.1f} hours")

# Phase 2: Bayesian Optimization
bo_evals = N_ITERATIONS * BATCH_SIZE
bo_min = bo_evals * MIN_PER_EVAL
bo_avg = bo_evals * AVG_PER_EVAL
bo_max = bo_evals * MAX_PER_EVAL

print(f"" + "="*80)
print("PHASE 2: Bayesian Optimization")
print("="*80)
print(f"Iterations: {N_ITERATIONS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Evaluations: {bo_evals}")
print(f"Estimated time:")
print(f"  Optimistic:  {bo_min:>4} minutes = {bo_min/60:.1f} hours = {bo_min/60/24:.1f} days")
print(f"  Realistic:   {bo_avg:>4} minutes = {bo_avg/60:.1f} hours = {bo_avg/60/24:.1f} days")
print(f"  Pessimistic: {bo_max:>4} minutes = {bo_max/60:.1f} hours = {bo_max/60/24:.1f} days")

# Total time
total_min = initial_min + bo_min
total_avg = initial_avg + bo_avg
total_max = initial_max + bo_max

print(f"" + "="*80)
print("TOTAL TIME ESTIMATE")
print("="*80)
print(f"Total evaluations: {TOTAL_BUDGET}")
print(f"Estimated total time:")
print(f"  Optimistic:  {total_min:>5} minutes = {total_min/60:>5.1f} hours = {total_min/60/24:>4.1f} days")
print(f"  Realistic:   {total_avg:>5} minutes = {total_avg/60:>5.1f} hours = {total_avg/60/24:>4.1f} days [EXPECTED]")
print(f"  Pessimistic: {total_max:>5} minutes = {total_max/60:>5.1f} hours = {total_max/60/24:>4.1f} days")

# Expected completion time
from datetime import datetime, timedelta
start_time = datetime.now()
expected_completion_min = start_time + timedelta(minutes=total_min)
expected_completion_avg = start_time + timedelta(minutes=total_avg)
expected_completion_max = start_time + timedelta(minutes=total_max)

print(f"If started now ({start_time.strftime('%Y-%m-%d %H:%M')}):")
print(f"  Optimistic completion:  {expected_completion_min.strftime('%Y-%m-%d %H:%M')}")
print(f"  Realistic completion:   {expected_completion_avg.strftime('%Y-%m-%d %H:%M')} [EXPECTED]")
print(f"  Pessimistic completion: {expected_completion_max.strftime('%Y-%m-%d %H:%M')}")

print(f"" + "="*80)
print("NOTES")
print("="*80)
print("* Estimates assume consistent GPU/CPU availability")
print("* Actual time varies based on:")
print("  - Early stopping behavior (depends on hyperparameters)")
print("  - Hardware performance (GPU model, CPU cores)")
print("  - System load and other processes")
print("  - Data loading speed (disk I/O)")
print("* Checkpoints are saved every", CHECKPOINT_FREQ, "iterations")
print("* You can resume from any checkpoint if interrupted")
print("[!] RECOMMENDATION: Start with a small test run first!")
print(f"    Test config: N_INITIAL=3, N_ITERATIONS=2, BATCH_SIZE=2")
print(f"    Test time:   ~{3*AVG_PER_EVAL + 4*AVG_PER_EVAL:.0f} minutes = {(3*AVG_PER_EVAL + 4*AVG_PER_EVAL)/60:.1f} hours")
print("="*80)

# Ask user to confirm
if total_avg > 600:  # More than 10 hours
    print("[!] WARNING: This optimization will take a LONG time!")
    print("    Make sure you have:")
    print("    - Stable power supply")
    print("    - Reliable internet (if using remote GPU)")
    print("    - Sufficient disk space for checkpoints")
    print("    - Considered running a test first")


TIME ESTIMATION
Device: CPU
Estimated time per evaluation:
  Optimistic:  60 minutes
  Realistic:   120 minutes (expected)
  Pessimistic: 180 minutes
PHASE 1: Initial Sobol Sampling
Evaluations: 1
Estimated time:
  Optimistic:    60 minutes = 1.0 hours
  Realistic:    120 minutes = 2.0 hours
  Pessimistic:  180 minutes = 3.0 hours
PHASE 2: Bayesian Optimization
Iterations: 1
Batch size: 1
Evaluations: 1
Estimated time:
  Optimistic:    60 minutes = 1.0 hours = 0.0 days
  Realistic:    120 minutes = 2.0 hours = 0.1 days
  Pessimistic:  180 minutes = 3.0 hours = 0.1 days
TOTAL TIME ESTIMATE
Total evaluations: 2
Estimated total time:
  Optimistic:    120 minutes =   2.0 hours =  0.1 days
  Realistic:     240 minutes =   4.0 hours =  0.2 days [EXPECTED]
  Pessimistic:   360 minutes =   6.0 hours =  0.2 days
If started now (2026-08-13 16:00):
  Optimistic completion:  2026-08-13 18:00
  Realistic completion:   2026-08-13 20:00 [EXPECTED]
  Pessimistic completion: 2026-08-13 22:00
NOTES
* Es

C:\Users\HP\AppData\Local\Temp\ipykernel_21352\133490688.py:32: UserWarning: CPU training will be VERY slow! Highly recommend using GPU.
  warnings.warn(


## 7. Phase 1: Initial Sobol Sampling

Load or generate initial training data.

In [55]:
# Create output directory
output_dir = OUTPUT_DIR / RUN_ID
output_dir.mkdir(parents=True, exist_ok=True)

# Create incremental save callback
def save_evaluation_callback(hyperparams, metrics):
    """Save each evaluation immediately to CSV (incremental)"""
    csv_path = output_dir / 'evaluations.csv'
    
    # Create row
    row = {
        'learning_rate': hyperparams['learning_rate'],
        'weight_decay': hyperparams['weight_decay'],
        'dropout': hyperparams['dropout'],
        'augmentation_strength': hyperparams['augmentation_strength'],
        'unfreeze_fraction': hyperparams['unfreeze_fraction'],
        'pr_auc': metrics['pr_auc'],
        'auroc': metrics['auroc'],
        'brier': metrics['brier'],
        'cross_dataset_degradation': metrics['cross_dataset_degradation']  # UPDATED
    }
    
    # Append to CSV
    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode='w', header=True, index=False)
    
    print(f"  ✓ Saved to {csv_path.name}")

# ========================================
# RESUME LOGIC
# ========================================

if RESUME_FROM_CSV and CSV_RESUME_PATH is not None:
    # CSV RESUME (NEW!)
    csv_path = Path(CSV_RESUME_PATH)
    
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV resume enabled but file not found: {csv_path}\n"
            f"Please check your CSV_RESUME_PATH in Cell 8"
        )
    
    print("="*80)
    print("RESUMING FROM CSV (NO CHECKPOINT AVAILABLE)")
    print("="*80)
    print(f"Loading from: {csv_path}")

    X_train, Y_train, all_hyperparams, start_iter = resume_from_csv(
        csv_path=csv_path,
        transform=transform,
        output_dir_path=output_dir
    )

    print(f"\n✓ CSV resume complete")
    print(f"  Loaded from input: {csv_path}")
    print(f"  New outputs will go to: {output_dir}")
    print(f"  Ready to continue optimization from iteration {start_iter}")

elif RESUME_FROM is not None:
    # Manual resume from specified checkpoint
    print(f"Resuming from checkpoint: {RESUME_FROM}")
    gp_model, X_train, Y_train, start_iter, extras = \
        BoTorchCheckpoint.resume_from_checkpoint(RESUME_FROM, MultiObjectiveGPModel)
    all_hyperparams = extras.get('all_hyperparams', [])
    print(f"  Loaded {len(X_train)} evaluations")
    print(f"  Starting from iteration {start_iter}")
    
elif USE_KAGGLE_PERSISTENCE:
    # Try to download latest checkpoint from Kaggle Dataset
    print("="*80)
    print("CHECKING FOR EXISTING CHECKPOINTS IN KAGGLE DATASET")
    print("="*80)
    
    latest_checkpoint = kaggle_manager.download_latest_checkpoint()
    
    if latest_checkpoint is not None:
        print(f"\n✓ Found checkpoint: {latest_checkpoint.name}")
        
        # Validate checkpoint
        if kaggle_manager.validate_checkpoint(latest_checkpoint):
            print("  Checkpoint validation passed")
            
            # Resume from downloaded checkpoint
            gp_model, X_train, Y_train, start_iter, extras = \
                BoTorchCheckpoint.resume_from_checkpoint(latest_checkpoint, MultiObjectiveGPModel)
            all_hyperparams = extras.get('all_hyperparams', [])
            
            print(f"\n✓ Successfully resumed from Kaggle checkpoint")
            print(f"  Evaluations completed: {len(X_train)}")
            print(f"  Starting from iteration: {start_iter}")
            print(f"  Progress: {100 * len(X_train) / TOTAL_BUDGET:.1f}%")
            
            # Increment session count
            kaggle_manager.increment_session_count()
            session_num = kaggle_manager.get_session_count()
            print(f"  Session number: {session_num}")
        else:
            print("  ✗ Checkpoint validation failed")
            print("  Starting fresh with initial sampling")
            latest_checkpoint = None
    
    if latest_checkpoint is None:
        print("\nNo existing checkpoints found in Kaggle Dataset.")
        print("Starting fresh with initial Sobol sampling...")
        
        # Initialize first session
        kaggle_manager.increment_session_count()
        
        print("="*80)
        print("PHASE 1: INITIAL SOBOL SAMPLING")
        print("="*80)
        
        # Generate Sobol samples
        X_init = initial_sobol_sampling(
            n_vars=5,
            n_samples=N_INITIAL,
            bounds=bounds_linear
        )
        
        print(f"\nEvaluating {N_INITIAL} initial Sobol samples...")
        print("This will take approximately:", f"{N_INITIAL * 0.5:.1f} hours (30 min/eval)")
        
        # Evaluate initial samples WITH INCREMENTAL SAVING
        Y_init, hyperparams_init = evaluate_batch(
            evaluate_fn, X_init, transform, save_callback=save_evaluation_callback
        )
        
        # Initialize training data
        X_train = X_init
        Y_train = Y_init
        all_hyperparams = hyperparams_init
        start_iter = 0
        
        print(f"\n✓ Initial sampling complete")
        print(f"  Evaluations: {len(X_train)}")
        
else:
    # Standard single-session mode
    print("="*80)
    print("PHASE 1: INITIAL SOBOL SAMPLING")
    print("="*80)
    
    # Generate Sobol samples
    X_init = initial_sobol_sampling(
        n_vars=5,
        n_samples=N_INITIAL,
        bounds=bounds_linear
    )
    
    print(f"\nEvaluating {N_INITIAL} initial Sobol samples...")
    print("This will take approximately:", f"{N_INITIAL * 0.5:.1f} hours (30 min/eval)")
    
    # Evaluate initial samples WITH INCREMENTAL SAVING
    Y_init, hyperparams_init = evaluate_batch(
        evaluate_fn, X_init, transform, save_callback=save_evaluation_callback
    )
    
    # Initialize training data
    X_train = X_init
    Y_train = Y_init
    all_hyperparams = hyperparams_init
    start_iter = 0
    
    print(f"\n✓ Initial sampling complete")
    print(f"  Evaluations: {len(X_train)}")

print(f"\nAll evaluations saved to: {output_dir / 'evaluations.csv'}")
print(f"  File is updated after EVERY evaluation completes")

FileNotFoundError: CSV resume enabled but file not found: \kaggle\input\dataaa\evaluations (10).csv
Please check your CSV_RESUME_PATH in Cell 8

In [ ]:
# ========================================
# VERIFY FILE SAVING
# ========================================
print("\n" + "="*80)
print("FILE VERIFICATION")
print("="*80)

# Check directory exists
if output_dir.exists():
    print(f"✓ Output directory exists: {output_dir}")
    print(f"  Absolute path: {output_dir.resolve()}")
else:
    print(f"✗ Output directory NOT found: {output_dir}")

# List all files in output directory
if output_dir.exists():
    print(f"\nFiles in {output_dir}:")
    files = list(output_dir.iterdir())
    if files:
        for f in files:
            size_kb = f.stat().st_size / 1024
            print(f"  - {f.name} ({size_kb:.1f} KB)")
    else:
        print(f"  (directory is empty)")
else:
    print(f"\nCannot list files - directory does not exist")

# Check CSV file specifically
csv_path = output_dir / 'evaluations.csv'
if csv_path.exists():
    print(f"\n✓ evaluations.csv found!")
    print(f"  Path: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024:.1f} KB")
    
    # Preview first few lines
    try:
        df_check = pd.read_csv(csv_path)
        print(f"  Rows: {len(df_check)}")
        print(f"  Columns: {list(df_check.columns)}")
        print(f"\n  First few rows:")
        print(df_check.head().to_string(index=False))
    except Exception as e:
        print(f"  Error reading CSV: {e}")
else:
    print(f"\n✗ evaluations.csv NOT found at: {csv_path}")
    print(f"\n  Attempted to save to: {csv_path}")
    print(f"  But file does not exist!")
    
    # Try to find where it might have been saved
    print(f"\n  Checking current working directory:")
    cwd = Path.cwd()
    print(f"    Current dir: {cwd}")
    for f in cwd.glob("evaluations.csv"):
        print(f"    Found: {f}")

print("="*80)

In [ ]:
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

print("="*80)
print("PHASE 2: BAYESIAN OPTIMIZATION")
print("="*80)
print(f"\nRunning {N_ITERATIONS} iterations with batch size {BATCH_SIZE}")
print(f"Expected time: {N_ITERATIONS * BATCH_SIZE * 0.5:.1f} hours")

for iteration in range(start_iter, N_ITERATIONS):
    # ========================================
    # TIME BUDGET CHECK (KAGGLE MULTI-SESSION)
    # ========================================
    if USE_KAGGLE_PERSISTENCE and AUTO_STOP_ON_TIMEOUT:
        if session_manager.should_stop():
            print(f"\n{'='*80}")
            print("TIME BUDGET EXCEEDED - STOPPING SAFELY")
            print(f"{'='*80}")
            print(f"Completed iterations: {iteration}/{N_ITERATIONS}")
            print(f"Time elapsed: {session_manager.time_elapsed_hours():.1f} hours")
            print(f"Time remaining: {session_manager.time_remaining_hours():.1f} hours")
            print(f"Total evaluations: {len(X_train)}/{TOTAL_BUDGET}")
            
            print("\nSaving emergency checkpoint before stopping...")
            
            # Save emergency checkpoint
            checkpoint_path = output_dir / f'checkpoint_iter{iteration}_emergency.pt'
            BoTorchCheckpoint.save(
                checkpoint_path,
                gp_model,
                X_train,
                Y_train,
                iteration,
                all_hyperparams=all_hyperparams,
                config={
                    'n_initial': N_INITIAL,
                    'n_iterations': N_ITERATIONS,
                    'batch_size': BATCH_SIZE
                }
            )
            print(f"  ✓ Local checkpoint saved: {checkpoint_path.name}")
            
            # Upload to Kaggle Dataset
            print(f"  Uploading to Kaggle Dataset...")
            upload_success = kaggle_manager.upload_checkpoint(checkpoint_path)
            
            if upload_success:
                print(f"  ✓ Checkpoint uploaded to Kaggle Dataset")
            else:
                print(f"  ✗ Upload failed - checkpoint saved locally only")
            
            print("\n" + "="*80)
            print("TO RESUME IN NEXT SESSION:")
            print("="*80)
            print("1. Start a new Kaggle notebook session")
            print("2. Set USE_KAGGLE_PERSISTENCE = True")
            print("3. Run all cells - checkpoint will auto-download")
            print(f"4. Optimization will continue from iteration {iteration}")
            print(f"5. Remaining: {N_ITERATIONS - iteration} iterations")
            print("="*80)
            
            break  # Exit loop safely
    
    # ========================================
    # STANDARD BO ITERATION
    # ========================================
    print(f"\n{'='*80}")
    print(f"BO Iteration {iteration+1}/{N_ITERATIONS}")
    print(f"{'='*80}")
    
    # Show time progress if multi-session enabled
    if USE_KAGGLE_PERSISTENCE:
        elapsed = session_manager.time_elapsed_hours()
        remaining = session_manager.time_remaining_hours()
        print(f"Session time: {elapsed:.1f}h elapsed, {remaining:.1f}h remaining")
    
    # Update GP model
    print("\n[1/4] Fitting GP models...")
    # Y_train follows thesis minimization convention:
    # [-PR-AUC, -AUROC, Brier, Degradation]

    Y_botorch = -Y_train

    # BoTorch qNEHVI uses maximization convention:
    # [PR-AUC, AUROC, -Brier, -Degradation]

    gp_model.update_data(X_train, Y_botorch)
    gp_model.fit_models()
    print("      ✓ GPs fitted")
    
    # Compute reference point
    ref_point = compute_reference_point(Y_botorch, offset=REF_POINT_OFFSET)
    print(f"\n[2/4] Reference point: {ref_point.numpy()}")
    
    # Create acquisition function
    print(f"\n[3/4] Optimizing qNEHVI acquisition...")
    acq = qNEHVIAcquisition(
        model=gp_model.models,
        ref_point=ref_point,
        bounds=bounds_linear,
        X_baseline=X_train
    )
    
    # Generate candidates
    X_next = acq.optimize(
        q=BATCH_SIZE,
        num_restarts=ACQ_RESTARTS,
        raw_samples=512
    )
    
    print(f"      ✓ Generated {len(X_next)} candidates")
    
    # Evaluate candidates
    print(f"\n[4/4] Evaluating {BATCH_SIZE} candidates...")
    Y_next, hyperparams_next = evaluate_batch(evaluate_fn, X_next, transform)
    
    # Update training data
    X_train = torch.cat([X_train, X_next], dim=0)
    Y_train = torch.cat([Y_train, Y_next], dim=0)
    all_hyperparams.extend(hyperparams_next)
    
    # Statistics
    print(f"\nIteration {iteration+1} complete:")
    print(f"  Total evaluations: {len(X_train)}/{TOTAL_BUDGET}")
    
    # Find current Pareto front - FIXED
    nds = NonDominatedSorting()
    pareto_indices = nds.do(Y_train.numpy(), only_non_dominated_front=True)
    print(f"  Pareto front size: {len(pareto_indices)}")
    
    # Save results
    df = pd.DataFrame(all_hyperparams)
    df['pr_auc'] = -Y_train[:, 0].numpy()
    df['auroc'] = -Y_train[:, 1].numpy()
    df['brier'] = Y_train[:, 2].numpy()
    df['robustness'] = Y_train[:, 3].numpy()
    df.to_csv(output_dir / 'evaluations.csv', index=False)
    
    # Checkpoint
    if (iteration + 1) % CHECKPOINT_FREQ == 0:
        checkpoint_path = output_dir / f'checkpoint_iter{iteration+1}.pt'
        BoTorchCheckpoint.save(
            checkpoint_path,
            gp_model,
            X_train,
            Y_train,
            iteration+1,
            all_hyperparams=all_hyperparams,
            config={
                'n_initial': N_INITIAL,
                'n_iterations': N_ITERATIONS,
                'batch_size': BATCH_SIZE
            }
        )
        print(f"  ✓ Checkpoint saved: {checkpoint_path.name}")
        
        # Upload to Kaggle Dataset if enabled
        if USE_KAGGLE_PERSISTENCE:
            print(f"  Uploading checkpoint to Kaggle Dataset...")
            upload_success = kaggle_manager.upload_checkpoint(checkpoint_path)
            
            if upload_success:
                print(f"  ✓ Uploaded to: {KAGGLE_DATASET_SLUG}")
                
                # Cleanup old local checkpoints (keep last 3)
                kaggle_manager.cleanup_old_checkpoints(keep_last_n=3)
            else:
                print(f"  ✗ Upload failed - continuing with local checkpoint only")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)

In [ ]:
# ========================================
# SESSION PROGRESS REPORT
# ========================================

if USE_KAGGLE_PERSISTENCE:
    print("\n" + "="*80)
    print("SESSION PROGRESS REPORT")
    print("="*80)
    
    # Get current state
    current_iter = start_iter  # This should be the last completed iteration
    nds = NonDominatedSorting()
    pareto_idx_current = nds.do(Y_train.numpy(), only_non_dominated_front=True)
    
    # Time information
    elapsed = session_manager.time_elapsed_hours()
    remaining = session_manager.time_remaining_hours()
    total_completed = len(X_train)
    
    print(f"\nTime:")
    print(f"  Start:     {session_manager.get_start_time_str()}")
    print(f"  Current:   {session_manager.get_current_time_str()}")
    print(f"  Elapsed:   {elapsed:.1f} hours ({session_manager.time_elapsed_minutes():.0f} min)")
    print(f"  Remaining: {remaining:.1f} hours ({session_manager.time_remaining_minutes():.0f} min)")
    print(f"  Stop at:   {session_manager.get_expected_stop_time_str()}")
    print(f"  Limit:     {SESSION_MAX_HOURS} hours (+ {SESSION_BUFFER_MINUTES} min buffer)")
    
    print(f"\nProgress:")
    print(f"  Completed: {total_completed}/{TOTAL_BUDGET} evaluations ({100*total_completed/TOTAL_BUDGET:.1f}%)")
    print(f"  Pareto front: {len(pareto_idx_current)} solutions")
    
    evals_possible = session_manager.estimate_evaluations_left(avg_eval_time_minutes=30)
    iters_possible = session_manager.estimate_iterations_left(avg_eval_time_minutes=30, batch_size=BATCH_SIZE)
    
    print(f"\nEstimates for remaining session time:")
    print(f"  Can fit ~{evals_possible} more evaluations")
    print(f"  Can fit ~{iters_possible} more BO iterations")
    
    print(f"\nCheckpoints:")
    checkpoints = kaggle_manager.list_available_checkpoints()
    print(f"  Saved to Kaggle Dataset: {len(checkpoints)} checkpoints")
    if checkpoints:
        print(f"  Latest: {checkpoints[-1]}")
    
    session_num = kaggle_manager.get_session_count()
    print(f"\nSession:")
    print(f"  Current session number: {session_num}")
    print(f"  Dataset: {KAGGLE_DATASET_SLUG}")
    
    if session_manager.should_stop():
        print(f"\n⚠️  WARNING: Time budget exceeded! Should stop now.")
    else:
        print(f"\n✓ Session has {remaining:.1f} hours remaining")
    
    print("="*80)
else:
    print("\nMulti-session support is disabled.")
    print(f"Completed {len(X_train)}/{TOTAL_BUDGET} evaluations ({100*len(X_train)/TOTAL_BUDGET:.1f}%)")

In [ ]:
# Compute final Pareto front - FIXED
nds = NonDominatedSorting()
pareto_idx = nds.do(Y_train.numpy(), only_non_dominated_front=True)

X_pareto = X_train[pareto_idx]
Y_pareto = Y_train[pareto_idx]

print(f"Final Statistics:")
print(f"  Total evaluations: {len(X_train)}")
print(f"  Pareto solutions: {len(pareto_idx)}")

# Save final results
final_results = {
    'X_all': X_train,
    'Y_all': Y_train,
    'X_pareto': X_pareto,
    'Y_pareto': Y_pareto,
    'pareto_indices': pareto_idx,
    'all_hyperparams': all_hyperparams,
    'gp_model': gp_model,
    'config': {
        'n_initial': N_INITIAL,
        'n_iterations': N_ITERATIONS,
        'batch_size': BATCH_SIZE,
        'total_budget': TOTAL_BUDGET
    }
}

results_path = output_dir / 'final_results.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(final_results, f)

print(f"\n✓ Results saved to: {results_path}")

## 10. Display Pareto Front Solutions

In [ ]:
print("\nPareto Front Solutions:")
print("="*90)
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8} {'LR':>10} {'WD':>10}")
print("-"*90)

for i, (idx, y) in enumerate(zip(pareto_idx, Y_pareto)):
    pr_auc = -y[0].item()
    auroc = -y[1].item()
    brier = y[2].item()
    robust = y[3].item()
    
    hp = all_hyperparams[idx]
    lr = hp['learning_rate']
    wd = hp['weight_decay']
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f} {lr:>10.6f} {wd:>10.6f}")

print("="*90)

## 11. Visualization: Convergence

In [ ]:
# Load evaluation history
df = pd.read_csv(output_dir / 'evaluations.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PR-AUC
axes[0, 0].plot(df['pr_auc'], 'o-', alpha=0.6, label='Evaluations')
axes[0, 0].plot(df['pr_auc'].cummax(), 'r-', linewidth=2, label='Best so far')
axes[0, 0].set_xlabel('Evaluation')
axes[0, 0].set_ylabel('PR-AUC')
axes[0, 0].set_title('PR-AUC Convergence')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# AUROC
axes[0, 1].plot(df['auroc'], 'o-', alpha=0.6, label='Evaluations')
axes[0, 1].plot(df['auroc'].cummax(), 'r-', linewidth=2, label='Best so far')
axes[0, 1].set_xlabel('Evaluation')
axes[0, 1].set_ylabel('AUROC')
axes[0, 1].set_title('AUROC Convergence')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Brier Score
axes[1, 0].plot(df['brier'], 'o-', alpha=0.6, label='Evaluations')
axes[1, 0].plot(df['brier'].cummin(), 'r-', linewidth=2, label='Best so far')
axes[1, 0].set_xlabel('Evaluation')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('Brier Score Convergence (lower is better)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Robustness Degradation
axes[1, 1].plot(df['robustness'], 'o-', alpha=0.6, label='Evaluations')
axes[1, 1].plot(df['robustness'].cummin(), 'r-', linewidth=2, label='Best so far')
axes[1, 1].set_xlabel('Evaluation')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('Robustness Degradation (lower is better)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Convergence plot saved to: {output_dir / 'convergence.png'}")

## 12. Visualization: Pareto Front (2D Projections)

In [ ]:
# Extract objectives
pr_auc_all = -Y_train[:, 0].numpy()
auroc_all = -Y_train[:, 1].numpy()
brier_all = Y_train[:, 2].numpy()
robust_all = Y_train[:, 3].numpy()

pr_auc_pareto = -Y_pareto[:, 0].numpy()
auroc_pareto = -Y_pareto[:, 1].numpy()
brier_pareto = Y_pareto[:, 2].numpy()
robust_pareto = Y_pareto[:, 3].numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# PR-AUC vs AUROC
axes[0, 0].scatter(pr_auc_all, auroc_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 0].scatter(pr_auc_pareto, auroc_pareto, c='red', s=100, marker='*', 
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 0].set_xlabel('PR-AUC')
axes[0, 0].set_ylabel('AUROC')
axes[0, 0].set_title('PR-AUC vs AUROC')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# PR-AUC vs Brier
axes[0, 1].scatter(pr_auc_all, brier_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 1].scatter(pr_auc_pareto, brier_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 1].set_xlabel('PR-AUC')
axes[0, 1].set_ylabel('Brier Score')
axes[0, 1].set_title('PR-AUC vs Brier')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# PR-AUC vs Robustness
axes[0, 2].scatter(pr_auc_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 2].scatter(pr_auc_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 2].set_xlabel('PR-AUC')
axes[0, 2].set_ylabel('Robustness Degradation')
axes[0, 2].set_title('PR-AUC vs Robustness')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# AUROC vs Brier
axes[1, 0].scatter(auroc_all, brier_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 0].scatter(auroc_pareto, brier_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 0].set_xlabel('AUROC')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('AUROC vs Brier')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# AUROC vs Robustness
axes[1, 1].scatter(auroc_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 1].scatter(auroc_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 1].set_xlabel('AUROC')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('AUROC vs Robustness')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Brier vs Robustness
axes[1, 2].scatter(brier_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 2].scatter(brier_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 2].set_xlabel('Brier Score')
axes[1, 2].set_ylabel('Robustness Degradation')
axes[1, 2].set_title('Brier vs Robustness')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'pareto_front_2d.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Pareto front visualization saved to: {output_dir / 'pareto_front_2d.png'}")

## 13. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("OPTIMIZATION SUMMARY")
print("="*80)

print(f"\nTotal evaluations: {len(X_train)}")
print(f"Pareto front size: {len(pareto_idx)}")

print(f"\nBest single-objective solutions:")
print(f"  Best PR-AUC:    {pr_auc_all.max():.4f}")
print(f"  Best AUROC:     {auroc_all.max():.4f}")
print(f"  Best Brier:     {brier_all.min():.4f}")
print(f"  Best Robustness: {robust_all.min():.4f}")

print(f"\nPareto front statistics:")
print(f"  PR-AUC range:    [{pr_auc_pareto.min():.4f}, {pr_auc_pareto.max():.4f}]")
print(f"  AUROC range:     [{auroc_pareto.min():.4f}, {auroc_pareto.max():.4f}]")
print(f"  Brier range:     [{brier_pareto.min():.4f}, {brier_pareto.max():.4f}]")
print(f"  Robustness range: [{robust_pareto.min():.4f}, {robust_pareto.max():.4f}]")

print(f"\nOutput files:")
print(f"  Evaluations CSV: {output_dir / 'evaluations.csv'}")
print(f"  Final results:   {output_dir / 'final_results.pkl'}")
print(f"  Convergence plot: {output_dir / 'convergence.png'}")
print(f"  Pareto plot:     {output_dir / 'pareto_front_2d.png'}")

## 14. Export Pareto Solutions for Further Analysis

In [ ]:
# Create DataFrame with Pareto solutions
pareto_solutions = []

for i, (idx, y) in enumerate(zip(pareto_idx, Y_pareto)):
    hp = all_hyperparams[idx]
    
    solution = {
        'solution_id': i,
        'learning_rate': hp['learning_rate'],
        'weight_decay': hp['weight_decay'],
        'dropout': hp['dropout'],
        'augmentation_strength': hp['augmentation_strength'],
        'unfreeze_fraction': hp['unfreeze_fraction'],
        'pr_auc': -y[0].item(),
        'auroc': -y[1].item(),
        'brier': y[2].item(),
        'robustness_degradation': y[3].item()
    }
    pareto_solutions.append(solution)

pareto_df = pd.DataFrame(pareto_solutions)
pareto_df.to_csv(output_dir / 'pareto_solutions.csv', index=False)

print("\nPareto Solutions:")
print(pareto_df.to_string(index=False))
print(f"\n✓ Pareto solutions saved to: {output_dir / 'pareto_solutions.csv'}")

## 15. Select Best Solution (by user preference)

Choose a solution from the Pareto front based on your preference.

In [ ]:
# Example: Select solution with best PR-AUC
best_pr_auc_idx = pr_auc_pareto.argmax()
best_solution = pareto_solutions[best_pr_auc_idx]

print("\nRecommended Solution (Best PR-AUC):")
print("="*50)
for key, value in best_solution.items():
    if key == 'solution_id':
        print(f"{key}: {value}")
    elif key in ['learning_rate', 'weight_decay']:
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value:.4f}")

print("\nUse these hyperparameters for training your final model!")

## ⚠️ IMPORTANT: Files for Zero-Shot Evaluation

**Before running the zero-shot evaluation notebook, you need these files:**

1. **INbreast Split Info (CRITICAL!):**
   - File: `{output_dir}/inbreast_split_info.json`
   - Created in: Cell 13 of this notebook
   - Contains: Calibration/test split used during optimization
   - **Why needed:** Zero-shot notebook must use the SAME split for comparable results

2. **Pareto Solutions:**
   - File: `{output_dir}/pareto_solutions.csv`
   - Created in: Cell 41 of this notebook
   - Contains: All non-dominated hyperparameter configurations

3. **Best Model Hyperparameters:**
   - Use the recommended solution from Cell 43
   - Update `BEST_HYPERPARAMS` in the zero-shot notebook

**To use in zero-shot notebook:**
1. Upload `inbreast_split_info.json` to Kaggle Dataset (if using Kaggle)
2. Update `INBREAST_SPLIT_JSON` path in zero-shot notebook Cell 4
3. The notebook will verify the split matches before training

---

## Next Steps

1. **Analyze Pareto front** - Choose solution based on your preference (trade-off between objectives)
2. **Zero-shot evaluation** - Evaluate selected solution(s) on INbreast dataset using `zeroshot_evaluation_updated.ipynb`
3. **Train final model** - Use selected hyperparameters for full training
4. **Compare with baselines** - If you have NSGA-III results, compare Pareto fronts

## Notes

- **Resuming:** Set `RESUME_FROM` variable to checkpoint path
- **Shorter runs:** Reduce `N_ITERATIONS` and `BATCH_SIZE` for testing
- **Parallel evaluation:** Current implementation is sequential; can be parallelized with multi-GPU
- **GP refitting:** GPs are refitted every iteration for maximum accuracy